In [ ]:
pip install --upgrade pip

In [ ]:
pip install pandas scipy matplotlib tqdm torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pickle
import pandas as pd
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import math
import scipy.special
import random as rd
import torch.nn.functional as F
import torchvision.models as models
import matplotlib.pyplot as plt
from torchvision.models import VGG16_Weights
from tqdm import tqdm
import pickle
import torch.optim.lr_scheduler as lr_scheduler
from sgr_utils import *

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: NVIDIA GeForce RTX 4060


### Loading cifar-10 dataset

In [5]:
# Define transforms for CIFAR-10 dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]) # imageNet stats by rgb channel

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root="../CIFAR/cifar_data", train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root="../CIFAR/cifar_data", train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=5, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=5, shuffle=False)

100%|██████████| 170M/170M [15:08<00:00, 188kB/s]    


### Training vgg16 for cifar-10 classification task

In [14]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load Pretrained VGG16 Model
vgg16 = models.vgg16(weights=VGG16_Weights.DEFAULT)
# Modify classifier for CIFAR-10 (10 classes)
vgg16.classifier[6] = nn.Linear(4096, 10)
# Move model to GPU if available
vgg16 = vgg16.to(device)

In [ ]:
# Define Loss, Optimizer and lr scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(vgg16.parameters(), lr=1e-3)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2)

# Training Loop
num_epochs = 50
for epoch in range(num_epochs):
    vgg16.train()
    running_loss = 0

    print('TRAINING EPOCH', epoch)
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = vgg16(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

    # Evaluate Model at the end of current epoch
    vgg16.eval()
    correct = 0
    total = 0
    print('TESTING')
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = vgg16(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

    scheduler.step(correct)
    torch.save(vgg16.state_dict(), "vgg16_cifar10.pth")

91.4 test accuracy after ~30 epochs with lr 1e-6, does not progress much => training stopped

### Retrieving Softmax Response, Predicted class and True class for all samples in train and in test sets

In [6]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load Pretrained VGG16 Model
vgg16 = models.vgg16(weights=VGG16_Weights.DEFAULT)
# Modify classifier for CIFAR-10 (10 classes)
vgg16.classifier[6] = nn.Linear(4096, 10)
# Move model to GPU if available
vgg16 = vgg16.to(device)

checkpoint = torch.load("../CIFAR/vgg16_cifar10.pth")
vgg16.load_state_dict(checkpoint)

<All keys matched successfully>

In [17]:
sgr_dico_train_set = prepare_sgr_dico(train_loader, model = vgg16, device = device, T = 70) # temperature scaling with T=70 to soften the probabilities
sgr_train_df = pd.DataFrame(sgr_dico_train_set)
pickle.dump(sgr_train_df, open('sgr_train_df','wb'))

100%|██████████| 10000/10000 [03:57<00:00, 42.18it/s]


In [18]:
sgr_dico_test_set = prepare_sgr_dico(test_loader, model = vgg16, device = device, T = 70)
sgr_test_df = pd.DataFrame(sgr_dico_test_set)
pickle.dump(sgr_test_df, open('sgr_test_df','wb'))

100%|██████████| 2000/2000 [00:47<00:00, 42.27it/s]
